# NumPy ↔ Pandas ↔ PyTorch — conversiones

Cheat sheet para pasar datos entre **arrays NumPy**, **DataFrames/Series de Pandas** y **tensores de PyTorch** sin perder forma ni tipo.

## Mapa rápido de conversiones

| Desde → Hacia | Cómo | ¿Copia memoria? |
|---------------|------|------------------|
| NumPy → Pandas | `pd.DataFrame(arr)`, `pd.Series(arr)` | Nueva estructura (comparte buffer en muchos casos) |
| Pandas → NumPy | `df.to_numpy()`, `s.to_numpy()` | Vista o copia según dtypes |
| NumPy → Tensor | `torch.from_numpy(arr)` | **Comparte** memoria (CPU) |
| NumPy → Tensor | `torch.tensor(arr)` | **Copia** |
| Tensor → NumPy | `t.detach().cpu().numpy()` | Copia si estaba en GPU o con grad |
| Pandas → Tensor | `torch.tensor(df.values)` o vía NumPy | Copia (solo numérico) |
| Tensor → Pandas | `pd.DataFrame(t.cpu().numpy())` | Copia |

**Regla práctica ML:** carga con Pandas → `to_numpy()` → `torch.tensor()` para entrenar; al volver, `tensor.cpu().numpy()` → DataFrame si necesitas etiquetas de columnas.

Requisitos: `pip install pandas numpy torch`

In [ ]:
import numpy as np
import pandas as pd
import torch

## Verificar tipos de datos

| Librería | Inspeccionar **clase del objeto** | Inspeccionar **dtype** (tipo de los valores) |
|----------|-----------------------------------|---------------------------------------------|
| Python | `type(x)` | N/A (tipos dinámicos por elemento) |
| NumPy | `type(arr)` → `ndarray` | `arr.dtype` |
| Pandas | `type(df)` → `DataFrame` | `df.dtypes`, `df['col'].dtype` |
| PyTorch | `type(t)` → `Tensor` | `t.dtype`, `t.shape` |

In [ ]:
arr = np.array([1, 2, 3], dtype=np.float32)
df = pd.DataFrame({"a": [1, 2], "b": [1.0, 2.0]})
t = torch.tensor([1.0, 2.0])

type(arr), arr.dtype          # (<class 'numpy.ndarray'>, dtype('float32'))
type(df), df.dtypes           # DataFrame + dtype por columna
type(t), t.dtype, t.shape     # (<class 'torch.Tensor'>, torch.float32, torch.Size([2]))

## 1. NumPy ↔ Pandas

Pandas envuelve arrays NumPy añadiendo **índice** (filas) y **nombres de columnas**.

In [ ]:
# --- NumPy → Pandas ---
arr = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])

df = pd.DataFrame(arr, columns=["x1", "x2"], index=["a", "b", "c"])
s = pd.Series(arr[:, 0], index=["a", "b", "c"], name="x1")

# Desde array 1D sin índice explícito → RangeIndex 0,1,2...
pd.DataFrame(np.random.randn(4, 3))

# --- Pandas → NumPy ---
df.to_numpy()              # Recomendado (pandas ≥ 0.24)
df.values                  # Atajo legacy; mismo rol en la práctica
s.to_numpy()               # Serie → array 1D

# Solo columnas numéricas (útil antes de tensor)
df.select_dtypes(include=[np.number]).to_numpy()

# Con dtype concreto
df.to_numpy(dtype=np.float32)

# Recuperar índice y columnas por separado (no van en el array)
df.index.tolist()
df.columns.tolist()

## 2. NumPy ↔ PyTorch

| Función | Uso |
|---------|-----|
| `torch.from_numpy(a)` | Tensor que **comparte** memoria con `a` (solo CPU, float/int compatibles) |
| `torch.tensor(a)` | **Copia** los datos; sirve para GPU, dtypes raros y arrays no modificables |
| `t.numpy()` | Tensor CPU sin grad → array (comparte si es contiguo y float) |
| `t.detach().cpu().numpy()` | Seguro en entrenamiento (corta el grafo y trae a CPU) |

In [ ]:
arr = np.array([[1.0, 2.0], [3.0, 4.0]], dtype=np.float32)

# --- NumPy → Tensor ---
t_shared = torch.from_numpy(arr)   # Misma memoria: cambiar t_shared modifica arr (y viceversa)
t_copy = torch.tensor(arr)         # Copia independiente
t_gpu = torch.tensor(arr, device="cuda")  # Solo si tienes GPU

t_shared.shape   # torch.Size([2, 2]) — igual que arr.shape

# --- Tensor → NumPy ---
t = torch.tensor([[10.0, 20.0], [30.0, 40.0]])
t.numpy()                          # OK si está en CPU y sin gradiente

t_grad = torch.tensor([1.0, 2.0], requires_grad=True)
loss = t_grad.sum()
loss.backward()
# t_grad.numpy()                   # Error: tensor con grad
t_grad.detach().numpy()            # Correcto

t_cuda = torch.tensor([1.0, 2.0], device="cuda") if torch.cuda.is_available() else t
# t_cuda.numpy()                   # Error si está en GPU
t_cuda.detach().cpu().numpy()      # Siempre seguro


## 3. Pandas ↔ PyTorch

No hay conversión directa: siempre pasas por **NumPy** (o lista) y pieres/restauras índice y nombres de columnas a mano.

Solo columnas **numéricas** pueden ir a tensor (objetos/categorías hay que codificarlas antes).

In [ ]:
df = pd.DataFrame(
    {"edad": [25, 30, 35], "ventas": [100.0, 150.0, 200.0]},
    index=["ana", "luis", "carla"],
)

# --- Pandas → Tensor ---
X = torch.tensor(df.to_numpy(dtype=np.float32))     # shape (n_filas, n_cols)
y = torch.tensor(df["ventas"].to_numpy())         # vector 1D desde una columna

# Alternativa equivalente
X2 = torch.from_numpy(df.select_dtypes(include=[np.number]).to_numpy())

# --- Tensor → Pandas ---
t = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
df_from_t = pd.DataFrame(
    t.detach().cpu().numpy(),
    columns=["x1", "x2"],
    index=["a", "b", "c"],
)

# Predicciones del modelo (1D) → Serie con el índice original
pred = torch.tensor([0.1, 0.2, 0.3])
pd.Series(pred.detach().cpu().numpy(), index=df.index, name="pred")

## 4. Tabla completa (todas las rutas)

```
                    ┌─────────────┐
                    │   Pandas    │
                    │ DataFrame / │
                    │   Series    │
                    └──────┬──────┘
           to_numpy()   │      │ pd.DataFrame(arr)
           .values      │      │ pd.Series(arr)
                        ▼      ▲
                    ┌─────────────┐
                    │    NumPy    │
                    │   ndarray   │
                    └──────┬──────┘
      from_numpy()      │      │ torch.tensor()
      (comparte)        │      │ (copia)
                        ▼      ▲
                    ┌─────────────┐
                    │   PyTorch   │
                    │   Tensor    │
                    └─────────────┘
           Pandas → Tensor:  torch.tensor(df.to_numpy())
           Tensor → Pandas:  pd.DataFrame(t.cpu().numpy(), columns=..., index=...)
           Tensor → NumPy:    t.detach().cpu().numpy()
           NumPy → Tensor:    torch.from_numpy(arr)  o  torch.tensor(arr)
```

## 5. Flujo típico en un proyecto de ML

1. **CSV → Pandas** (`read_csv`) — EDA, limpieza, nombres de columnas.
2. **Pandas → NumPy** — matriz `X` e vector `y` solo numéricos.
3. **NumPy → Tensor** — `torch.tensor(..., dtype=torch.float32)` para el `DataLoader`.
4. Tras `model(x)` — **Tensor → NumPy/Pandas** para métricas sklearn o gráficos.

In [ ]:
# Ejemplo mínimo de pipeline
df = pd.DataFrame({
    "feature1": [1.0, 2.0, 3.0, 4.0],
    "feature2": [10.0, 20.0, 30.0, 40.0],
    "target": [0, 1, 0, 1],
})

feature_cols = ["feature1", "feature2"]
X_np = df[feature_cols].to_numpy(dtype=np.float32)
y_np = df["target"].to_numpy(dtype=np.float32)

X = torch.tensor(X_np)
y = torch.tensor(y_np)

# Simular predicción
with torch.no_grad():
    y_pred = (X.mean(dim=1) > 15).float()  # tensor ficticio

resultado = pd.DataFrame({
    "target": df["target"],
    "pred": y_pred.cpu().numpy(),
}, index=df.index)
resultado

## 6. Trampas frecuentes

| Problema | Solución |
|----------|----------|
| `from_numpy` tras modificar dtype de pandas | Usar `df.to_numpy()` justo antes de convertir |
| Tensor en GPU → `.numpy()` falla | `.cpu().numpy()` |
| Tensor con `requires_grad` → `.numpy()` falla | `.detach().cpu().numpy()` |
| Columnas `object` / strings en DataFrame | Codificar (one-hot, `astype`) antes del tensor |
| Índice perdido al pasar a tensor | Guardar `df.index` y reasignar al volver |
| `int64` en NumPy en algunos ops torch | `astype(np.float32)` o `torch.long()` para etiquetas enteras |
| Cambiar array NumPy compartido con `from_numpy` | Esperado: usar `torch.tensor` si quieres aislar |

In [ ]:
# Tipos de etiquetas en clasificación
y_int = df["target"].to_numpy()           # int64 de pandas
y_long = torch.tensor(y_int, dtype=torch.long)   # esperado por CrossEntropyLoss
y_float = torch.tensor(y_int, dtype=torch.float32)  # esperado por BCELoss / regresión

# Batch: añadir dimensión si hace falta (ej. imagen CHW, secuencias)
x = torch.tensor([[1.0, 2.0]])   # shape (1, 2) = un batch de 2 features
x.unsqueeze(0)                 # (1, 1, 2) — ejemplo de dim extra
x.squeeze()                    # quita dims de tamaño 1